# Probability distribution of the sum of square-wave amplitudes

Each of $N$ independent square waves is at **100%** amplitude with probability $p$ and at **50%** amplitude with probability $1-p$. The waves run forever and switch independently, so at any instant the amplitudes are i.i.d.

**Goal:** find the probability distribution of the total amplitude

$$S = \sum_{i=1}^{N} X_i$$

where each $X_i \in \{A_{\text{high}}, A_{\text{low}}\}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Inputs

In [ ]:
# --- user inputs ---
N_WAVES = 30          # number of independent square waves
P_HIGH = 0.8            # fraction of time each wave is at high amplitude
A_HIGH = 1.0            # high amplitude (100%)
A_LOW = 0.5             # low amplitude (50%)

# Monte Carlo settings (for validation / visualization)
N_SAMPLES = 500_000
RNG = np.random.default_rng(42)

## Exact distribution

Let $K$ be the number of waves at the **low** amplitude. Then

$$K \sim \mathrm{Binomial}(N,\, 1-p)$$

and the sum is deterministic given $K$:

$$S = K\,A_{\text{low}} + (N-K)\,A_{\text{high}} = N\,A_{\text{high}} - K\,(A_{\text{high}} - A_{\text{low}})$$

So $S$ takes $N+1$ equally spaced values, one for each $K \in \{0, 1, \ldots, N\}$.

In [ ]:
p_low = 1.0 - P_HIGH
delta = A_HIGH - A_LOW

k = np.arange(N_WAVES + 1)
s_values = N_WAVES * A_HIGH - k * delta
probabilities = stats.binom.pmf(k, N_WAVES, p_low)

assert np.isclose(probabilities.sum(), 1.0)

mean_s = probabilities @ s_values
var_s = probabilities @ (s_values - mean_s) ** 2

print(f"Low-amplitude probability (per wave): {p_low:.2f}")
print(f"Support: S ∈ {{{s_values.min():.1f}, ..., {s_values.max():.1f}}}  ({len(s_values)} points, step = {delta:.2f})")
print(f"E[S] = {mean_s:.4f}")
print(f"Var(S) = {var_s:.4f}")
print(f"Std(S) = {np.sqrt(var_s):.4f}")

In [ ]:
summary = np.column_stack([k, s_values, probabilities])
print(f"{'K (low)':>8} {'S':>8} {'P(S)':>12}")
print("-" * 32)
for row in summary:
    print(f"{int(row[0]):8d} {row[1]:8.1f} {row[2]:12.6f}")

## Monte Carlo check

Sample $N$ wave amplitudes at a random instant and compare the empirical histogram to the exact pmf.

In [ ]:
samples = RNG.choice([A_HIGH, A_LOW], size=(N_SAMPLES, N_WAVES), p=[P_HIGH, p_low]).sum(axis=1)

mc_counts = np.bincount(np.rint(samples / delta).astype(int), minlength=N_WAVES + 1)
mc_probs = mc_counts / N_SAMPLES

max_abs_err = np.max(np.abs(mc_probs - probabilities))
print(f"Max |MC - exact| over support: {max_abs_err:.5f}")

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bar_width = delta * 0.85
ax.bar(s_values, probabilities, width=bar_width, alpha=0.7, label="Exact pmf", edgecolor="C0")
ax.bar(s_values, mc_probs, width=bar_width * 0.5, alpha=0.5, label=f"MC ({N_SAMPLES:,} samples)", edgecolor="C1")

ax.axvline(mean_s, color="k", ls="--", lw=1, label=f"E[S] = {mean_s:.2f}")
ax.set_xlabel("Total amplitude S")
ax.set_ylabel("Probability")
ax.set_title(f"Distribution of sum of {N_WAVES} square waves (p_high={P_HIGH})")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Cumulative distribution

In [ ]:
cdf = np.cumsum(probabilities)

fig, ax = plt.subplots(figsize=(10, 4))
ax.step(s_values, cdf, where="post", label="Exact CDF")
ax.set_xlabel("Total amplitude S")
ax.set_ylabel("P(S ≤ s)")
ax.set_title("Cumulative distribution of total amplitude")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Closed-form moments (optional sanity check)

Per wave: $E[X_i] = p\,A_{\text{high}} + (1-p)\,A_{\text{low}}$ and
$\mathrm{Var}(X_i) = p(1-p)(A_{\text{high}} - A_{\text{low}})^2$.

By independence: $E[S] = N\,E[X_i]$ and $\mathrm{Var}(S) = N\,\mathrm{Var}(X_i)$.

In [ ]:
ex_i = P_HIGH * A_HIGH + p_low * A_LOW
var_x_i = P_HIGH * p_low * delta ** 2

print(f"E[X_i]  = {ex_i:.4f}   →  E[S]  = {N_WAVES * ex_i:.4f}  (matches {mean_s:.4f})")
print(f"Var(X_i) = {var_x_i:.4f}   →  Var(S) = {N_WAVES * var_x_i:.4f}  (matches {var_s:.4f})")